The attached files are a collection of tweets labelled with sentiment in 3 categories:

sentiments = {
    "LABEL_0": "Bearish", 
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}  

Train a LSTM network with the training file. Validate the trained model with the valid file. Comment what you are doing in each part of your code. As the better the code, comments and result validation as the better the grade.

In this notebook I'm going to build a model that can read a financial tweet and decide if the sentiment is bearish (label 0), bullish (label 1), or neutral (label 2).  
I'll use a Long Short‑Term Memory (LSTM) network because tweets are sequences of words, and LSTMs are great at understanding the order and context in text. 

# 1. Importing the tools I'll need

I'll use pandas to read the CSV files, re to clean the tweets, collections.Counter to count words, and of course torch and torch.nn for the neural network.

In [ ]:
import pandas as pd
import re
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# I'll check if a GPU is available, if yes, I'll use it because it's much faster.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("I'm using:", device)


I'm using: cpu


# 2. Loading the data
The files sent_train.csv and sent_valid.csv contain the tweets and their labels.  
I'll load them with pandas and take a quick look to make sure everything is fine.

In [3]:
train_df = pd.read_csv('sent_train.csv')
valid_df = pd.read_csv('sent_valid.csv')

print("Training set size:", len(train_df))
print("Validation set size:", len(valid_df))
print("\nFirst 5 rows of training data:")
print(train_df.head())
print("\nLabel distribution in training:")
print(train_df['label'].value_counts())

Training set size: 9543
Validation set size: 2388

First 5 rows of training data:
                                                text  label
0  $BYND - JPMorgan reels in expectations on Beyo...      0
1  $CCL $RCL - Nomura points to bookings weakness...      0
2  $CX - Cemex cut at Credit Suisse, J.P. Morgan ...      0
3  $ESS: BTIG Research cuts to Neutral https://t....      0
4  $FNKO - Funko slides after Piper Jaffray PT cu...      0

Label distribution in training:
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


# 3. Text preprocessing, cleaning the tweets
Tweets often contain things like URLs and @mentions that don't carry much sentiment.  
I'll write a small function that removes them, turns everything to lowercase, and splits the tweet into words (tokens).  
This is a very basic tokenization, but it should work for this case.

In [4]:
def tokenize(text):
    """
    Clean a tweet:
    - remove URLs (http...)
    - remove @mentions
    - convert to lower case
    - split on whitespace
    """
    # remove URLs
    text = re.sub(r'http\S+', '', text)
    # remove mentions
    text = re.sub(r'@\w+', '', text)
    # lowercase and strip extra spaces
    text = text.lower().strip()
    # split into words
    tokens = text.split()
    return tokens

In [ ]:
# To test it i do it with one tweet to see if it works
example = train_df.iloc[0]['text']
print("Original tweet:", example)
print("Tokens after cleaning:", tokenize(example))

Original tweet: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Tokens after cleaning: ['$bynd', '-', 'jpmorgan', 'reels', 'in', 'expectations', 'on', 'beyond', 'meat']


# 4. Building the vocabulary
Now I need to create a mapping from each word to a number (index).  
I'll add two special tokens:
- "PAD" will used to make all tweets the same length.
- "UNK" that will represent words that appear in validation but not in training.

To keep the vocabulary size manageable, I'll only keep words that appear at least twice in the training data.

In [6]:
# Count all words in the training tweets
word_counts = Counter()
for text in train_df['text']:
    tokens = tokenize(text)
    word_counts.update(tokens)

# Set minimum frequency to 2
min_freq = 2
# Start with special tokens
vocab = ['<PAD>', '<UNK>']
# Add words that appear at least min_freq times
for word, count in word_counts.items():
    if count >= min_freq:
        vocab.append(word)

# Create the mapping word (index)
word2idx = {word: idx for idx, word in enumerate(vocab)}

print("Vocabulary size:", len(vocab))
print("First 20 words:", vocab[:20])

Vocabulary size: 8266
First 20 words: ['<PAD>', '<UNK>', '$bynd', '-', 'jpmorgan', 'reels', 'in', 'expectations', 'on', 'beyond', 'meat', '$ccl', '$rcl', 'nomura', 'points', 'to', 'bookings', 'weakness', 'at', 'carnival']


# 5. Turning tweets into lists of indices
I'll write a function that takes a tweet, tokenizes it, and replaces each word with its index.  
If a word is not in the vocabulary, I'll use the index of the UNK.

In [7]:
def encode(text):
    tokens = tokenize(text)
    ids = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]
    return ids

# Apply encoding to all tweets in train and validation
train_encoded = [encode(text) for text in train_df['text']]
valid_encoded = [encode(text) for text in valid_df['text']]

# This will show the length of the first encoded tweet
print("Length of first encoded tweet:", len(train_encoded[0]))

Length of first encoded tweet: 9


# 6. Padding sequences to the same length
LSTMs need all sequences in a batch to have the same length.  
I'll find the maximum length in the training set and then pad (or truncate) every tweet to that length.  
Padding means adding PAD tokens at the end.  
This is exactly what we did in Exercise 4 of Session 07 when we used pooling, but here i will keep the full sequence for the LSTM.

In [8]:
max_len = max(len(seq) for seq in train_encoded)
print("Maximum sequence length in training:", max_len)

def pad_sequence(seq, max_len):
    """If the sequence is longer than max_len, cut it; if shorter, add PAD tokens at the end."""
    if len(seq) >= max_len:
        return seq[:max_len]
    else:
        return seq + [word2idx['<PAD>']] * (max_len - len(seq))

train_padded = [pad_sequence(seq, max_len) for seq in train_encoded]
valid_padded = [pad_sequence(seq, max_len) for seq in valid_encoded]

print("Shape of training padded data:", len(train_padded), "x", len(train_padded[0]))

Maximum sequence length in training: 31
Shape of training padded data: 9543 x 31


# 7. Getting the labels
The labels are already numbers (0,1,2). I'll convert them to numpy arrays (they will become tensors later).

In [9]:
train_labels = train_df['label'].values
valid_labels = valid_df['label'].values

print("Labels distribution in training:", dict(zip(*np.unique(train_labels, return_counts=True))))

Labels distribution in training: {np.int64(0): np.int64(1442), np.int64(1): np.int64(1923), np.int64(2): np.int64(6178)}


# 8. Creating PyTorch Dataset and DataLoader
What i´m gonna do now follows literally the pattern from the notebook of the sesion 08.  
I'll make a custom Dataset that returns a (text, label) pair.  
Then I'll use DataLoader to get batches during the training.

In [10]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        # Convert to tensors (long type because they are indices)
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# I'll use a batch size of 64, because this fits in memory on my machine.
batch_size = 64

train_dataset = TweetDataset(train_padded, train_labels)
valid_dataset = TweetDataset(valid_padded, valid_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(valid_loader))


Number of training batches: 150
Number of validation batches: 38


# 9. Building the LSTM model
I would said that thi is the heart of my lab. Here I'll create a class that contains:
- embedding layer that turns word indices into dense vectors (like we saw in S07).
- an LSTM layer that reads the sequence of word vectors and produces a final hidden state.
- a linear layer that takes that hidden state and outputs scores for the three classes.
- dropout to prevent overfitting.

The architecture is very similar to the sentiment classifier of the exercise 3 of the sesion 8, but there we had 2 classes and here we have 3.  
I'm using batch_first=True so that the input shape is (batch, seq_len, features), i do it because i find it that easier to think about.

In [ ]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=1, dropout=0.5):
        super().__init__()
        # Embedding layer, basically its learns a vector for each word
        # I set padding_idx so that the PAD token always gives a zero vector and doesn't affect gradients
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=word2idx['<PAD>'])
        # LSTM layer
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        # Linear layer to map hidden state to class scores
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len)
        embedded = self.embedding(x)          # (batch, seq_len, embed_dim)
        # Run through LSTM
        lstm_out, (h_n, c_n) = self.lstm(embedded)   # h_n shape: (num_layers, batch, hidden_dim)
        # Take the last hidden state from the top layer
        last_hidden = h_n[-1]                 # (batch, hidden_dim)
        # Apply dropout (helps with generalization)
        last_hidden = self.dropout(last_hidden)
        # Compute logits (scores) for each class
        logits = self.fc(last_hidden)         # (batch, num_classes)
        return logits

# I'll choose some reasonable hyperparameters
vocab_size = len(vocab)
embed_dim = 100       # each word becomes a 100‑dimensional vector
hidden_dim = 128      # the LSTM hidden state size
num_classes = 3
num_layers = 1        # i think one LSTM layer is enough for a start
dropout = 0.3

model = LSTMSentiment(vocab_size, embed_dim, hidden_dim, num_classes, num_layers, dropout)
model.to(device)

# Now i´ll count the total number of parameters (just out of curiosity)
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters in the model:", total_params)

Total parameters in the model: 944747


# 10. Choosing loss function and optimizer 
For classification I'll use CrossEntropyLoss and that it's because is the standard choice for multi‑class problems.  
For the optimizer I'll pick Adam because it usually works well without much tuning.

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# I'll also keep lists to track losses and accuracies during training
num_epochs = 5
train_losses = []
train_accs = []
valid_losses = []
valid_accs = []

# 11. Training loop
This is where we use all that we are creating before. For each epoch:
- I set the model to training mode.
- I go through all batches in the training loader.
   - I move the data to the GPU (or CPU) and do a forward pass.
   - I compute the loss, then backpropagate and update weights.
   - I also count how many predictions were correct.
 - At the end of the epoch, I compute the average loss and accuracy for training.
 - Then I switch to evaluation mode and run the validation set (without gradients) to see how the model performs on unseen data.
 - I print everything so I can see if the model is improving.

In [15]:
for epoch in range(num_epochs):
    # Training phase
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)

        # Forward pass
        outputs = model(texts)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate statistics
        total_loss += loss.item() * texts.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = total_loss / len(train_dataset)
    epoch_train_acc = correct / total
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)

    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():   # no gradients needed for validation
        for texts, labels in valid_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * texts.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(valid_dataset)
    epoch_val_acc = val_correct / val_total
    valid_losses.append(epoch_val_loss)
    valid_accs.append(epoch_val_acc)

    # Print results for this epoch
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f}")
    print(f"  Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")


Epoch 1/5
  Train Loss: 0.5063 - Train Acc: 0.7800
  Val Loss: 0.6609 - Val Acc: 0.7412
Epoch 2/5
  Train Loss: 0.4523 - Train Acc: 0.7967
  Val Loss: 0.7119 - Val Acc: 0.7332
Epoch 3/5
  Train Loss: 0.4084 - Train Acc: 0.8092
  Val Loss: 0.7063 - Val Acc: 0.7270
Epoch 4/5
  Train Loss: 0.3573 - Train Acc: 0.8420
  Val Loss: 0.7374 - Val Acc: 0.7379
Epoch 5/5
  Train Loss: 0.3120 - Train Acc: 0.8793
  Val Loss: 0.6884 - Val Acc: 0.7747


# 12. Final evaluation
After training, I want to see the best validation accuracy and the final one.  
This tells me if the model is learning something useful.

In [17]:
print("Training finished")
print(f"Best validation accuracy: {max(valid_accs):.4f}")
print(f"Final validation accuracy: {valid_accs[-1]:.4f}")

# I could save the model if I wanted to use it later:
torch.save(model.state_dict(), 'lstm_sentiment.pth')

Training finished
Best validation accuracy: 0.7747
Final validation accuracy: 0.7747


# 13. Testing on some random tweets
Numbers are nice, but I also want to see with my own eyes what the model predicts.  
I'll pick 5 random tweets from the validation set and show the true label and the predicted label.


In [19]:
model.eval()
indices = np.random.choice(len(valid_df), size=5, replace=False)
label_names = {0: "RED Bearish", 1: "GREEN Bullish", 2: "WHITE Neutral"}

for idx in indices:
    text = valid_df.iloc[idx]['text']
    true_label = valid_df.iloc[idx]['label']
    
    # Process the tweet exactly as we did during training
    tokens = tokenize(text)
    ids = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]
    padded = pad_sequence(ids, max_len)
    input_tensor = torch.tensor([padded], dtype=torch.long).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).item()
    
    print("Tweet:", text[:100], "..." if len(text)>100 else "")
    print(f"  True label: {label_names[true_label]}")
    print(f"  Predicted : {label_names[pred]}")
    print()


Tweet: Playtech warns it will miss earnings expectations https://t.co/Zsz2VKkcrc 
  True label: RED Bearish
  Predicted : RED Bearish

Tweet: Nigeria won't seek a suspension of interest payments from its Eurobond bondholders, but will seek de ...
  True label: WHITE Neutral
  Predicted : WHITE Neutral

Tweet: Euro pinned near 10-day lows as outlook bleak - SI https://t.co/kPqleNJhIY 
  True label: RED Bearish
  Predicted : RED Bearish

Tweet: Tekla Healthcare Opportunities Fund declares $0.1125 dividend 
  True label: WHITE Neutral
  Predicted : WHITE Neutral

Tweet: $FISI - Financial Institutions, Inc. (FISI) CEO Marty Birmingham on Q4 2019 Results - Earnings Call  ...
  True label: WHITE Neutral
  Predicted : WHITE Neutral



In [ ]:
git add .
git commit -m "I do the final evaluation and i save it"
git push origin master